In [1]:
# PROYECTO 1: MONITOR DE RIESGO Y RENDIMIENTO DE CARTERA (REVISADO 2026)
# ==============================================================================

# 1. INSTALACIÓN Y IMPORTACIÓN
!pip install yfinance plotly --quiet
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.figure_factory as ff
import datetime

# 2. CONFIGURACIÓN DE ACTIVOS Y FECHAS
# Hemos elegido activos con diferentes niveles de riesgo para el análisis
activos = ['AAPL', 'MSFT', 'GOOGL', 'SPY', 'GLD']
fin = datetime.datetime.now()
inicio = fin - datetime.timedelta(days=3*365)

print(f"Descargando datos desde {inicio.date()} hasta {fin.date()}...")

# 3. EXTRACCIÓN DE DATOS (FIXED)
# Usamos group_by='column' para evitar errores de jerarquía en las columnas
raw_data = yf.download(activos, start=inicio, end=fin, group_by='column', auto_adjust=False)

# Verificación de la columna de precios ajustados
if 'Adj Close' in raw_data.columns:
    df_precios = raw_data['Adj Close']
else:
    df_precios = raw_data['Close']
    print("Aviso: Utilizando precios de cierre estándar.")

# 4. CÁLCULO DE MÉTRICAS FINANCIERAS
# Retornos diarios (limpiando el primer valor nulo)
retornos = df_precios.pct_change().dropna()

# Volatilidad Anualizada (Riesgo)
volatilidad_anual = (retornos.std() * np.sqrt(252)).sort_values(ascending=False)

# Retorno Acumulado (Crecimiento de $1)
retorno_acumulado = (1 + retornos).cumprod()

# Value at Risk (VaR) al 95%
var_95 = retornos.quantile(0.05)

# 5. VISUALIZACIÓN INTERACTIVA
print("\nGenerando gráficos...")

# Gráfico de Rendimiento
fig_rendimiento = px.line(retorno_acumulado,
                          title='Crecimiento de la Inversión (Base $1)',
                          labels={'value': 'Multiplicador', 'Date': 'Fecha'},
                          template='plotly_dark')
fig_rendimiento.show()

# Matriz de Correlación
corr_matrix = retornos.corr().round(2)
fig_corr = ff.create_annotated_heatmap(
    z=corr_matrix.values,
    x=list(corr_matrix.columns),
    y=list(corr_matrix.index),
    colorscale='Viridis'
)
fig_corr.update_layout(title='Matriz de Correlación de Activos')
fig_corr.show()

# 6. RESUMEN EJECUTIVO
print("\n" + "="*30)
print("RESUMEN DE RIESGO Y RETORNO")
print("="*30)
resumen = pd.DataFrame({
    'Volatilidad Anual': volatilidad_anual,
    'VaR Diario (95%)': var_95,
    'Retorno Total %': (retorno_acumulado.iloc[-1] - 1) * 100
})
print(resumen.round(4))

Descargando datos desde 2023-03-03 hasta 2026-03-02...


[*********************100%***********************]  5 of 5 completed



Generando gráficos...



RESUMEN DE RIESGO Y RETORNO
        Volatilidad Anual  VaR Diario (95%)  Retorno Total %
Ticker                                                      
AAPL               0.2573           -0.0248          74.5791
GLD                0.1885           -0.0157         185.5145
GOOGL              0.2917           -0.0246         224.6462
MSFT               0.2367           -0.0233          58.8307
SPY                0.1513           -0.0142          76.5367
